# Question 2

### Part 1

In [27]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

# Read input data from Excel
excel_file = '/Users/juanjo/Library/CloudStorage/OneDrive-UniversityofOklahoma/1 2025 Spring/Optimization/Homeworks/HW1/Transportation data.xlsx'

# Travel times
travel_time_df = pd.read_excel(excel_file, sheet_name='Travel times', index_col=0)
stations = list(travel_time_df.index)

travel_time = {}
for i in stations:
    for j in stations:
        val = travel_time_df.loc[i, j]
        if pd.notnull(val) and val != '-':
            travel_time[(i, j)] = float(val)

# OD matrix
od_df = pd.read_excel(excel_file, sheet_name='OD matrix', index_col=0)
demand = {}
for o in stations:
    for d in stations:
        if o != d:
            val = od_df.loc[o, d]
            if pd.notnull(val):
                demand[(o, d)] = float(val)

# Corridor assignments
corridor_assignments = {
    "53 Street": [1], "SB Park": [1], "66 Street": [1], "72 Street": [1, 2], 
    "80 Street": [2], "98 Street": [2], "Sub Avenue": [2], "100 Street": [2, 3], 
    "53 Avenue": [3], "19 Avenue": [3], "11 Avenue": [3]
}

# Expanded network nodes
expanded_nodes = []
for s in stations:
    for c in corridor_assignments[s]:
        expanded_nodes.append((s, c))
    expanded_nodes.append(("enter", s))
    expanded_nodes.append(("exit", s))

# Adjacency list with costs
edges = {}

# Travel time edges within corridors
for (i, j), t in travel_time.items():
    common_corridors = set(corridor_assignments[i]).intersection(set(corridor_assignments[j]))
    for c in common_corridors:
        edges[((i, c), (j, c))] = t

# Transfer arcs (5 minutes) at transshipment stations
for s in stations:
    if len(corridor_assignments[s]) > 1:
        for c1 in corridor_assignments[s]:
            for c2 in corridor_assignments[s]:
                if c1 < c2:
                    edges[((s, c1), (s, c2))] = 5.0
                    edges[((s, c2), (s, c1))] = 5.0

# Enter/exit arcs (2 minutes)
for s in stations:
    for c in corridor_assignments[s]:
        edges[(("enter", s), (s, c))] = 2.0
        edges[((s, c), ("exit", s))] = 2.0

# Create Gurobi model
model = gp.Model("TransitNetwork")

# Create variables
x = {}
for (o, d), dem in demand.items():
    if dem > 0:
        for (u, v) in edges.keys():
            x[(u, v, o, d)] = model.addVar(lb=0.0, ub=dem, name=f"x[{u}->{v},{o}->{d}]")

model.update()

# Flow balance constraints
for (o, d), dem in demand.items():
    if dem > 0:
        for n in expanded_nodes:
            inflow = gp.LinExpr()
            outflow = gp.LinExpr()

            for (u, v) in edges.keys():
                if v == n:
                    inflow.add(x[(u, v, o, d)])
                if u == n:
                    outflow.add(x[(u, v, o, d)])

            if n == ("enter", o):
                model.addConstr(outflow - inflow == dem)
            elif n == ("exit", d):
                model.addConstr(inflow - outflow == dem)
            else:
                model.addConstr(inflow == outflow)

# Objective function: minimize total travel time
model.setObjective(gp.quicksum(edges[(u, v)] * x[(u, v, o, d)] for (o, d) in demand for (u, v) in edges), GRB.MINIMIZE)

# Solve model
model.optimize()

# Output results
if model.status == GRB.OPTIMAL:
    print(f"Optimal objective value: {model.ObjVal:.2f} minutes")
else:
    print("Model did not solve optimally.")




Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 3850 rows, 8140 columns and 16280 nonzeros
Model fingerprint: 0x8277fde4
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+00, 5e+01]
  Bounds range     [3e+01, 2e+02]
  RHS range        [3e+01, 2e+02]
Presolve removed 2420 rows and 2820 columns
Presolve time: 0.01s
Presolved: 1430 rows, 5320 columns, 10640 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.0264000e+04   3.141500e+03   0.000000e+00      0s
     777    5.5604900e+05   0.000000e+00   0.000000e+00      0s

Solved in 777 iterations and 0.02 seconds (0.01 work units)
Optimal objective  5.560490000e+05
Optimal objective value: 556049.00 minutes


## Part 2

In [89]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd


# Read Excel data

excel_file = '/Users/juanjo/Library/CloudStorage/OneDrive-UniversityofOklahoma/1 2025 Spring/Optimization/Homeworks/HW1/Transportation data.xlsx'  # <-- Update as needed

# Read travel times (this must list actual travel times or '-' if no link)
travel_time_df = pd.read_excel(excel_file, sheet_name='Travel times', index_col=0)

# Read OD matrix
od_df = pd.read_excel(excel_file, sheet_name='OD matrix', index_col=0)

# Extract all stations from the OD matrix:
stations = list(od_df.index)

# Build demand dictionary
demand = {}
for o in stations:
    for d in stations:
        if o != d:
            val = od_df.loc[o, d]
            if pd.notnull(val):
                demand[(o, d)] = float(val)
            else:
                demand[(o, d)] = 0.0


# Define corridors (ordered lists of stations).

corridor1 = ["53 Street", "SB Park", "66 Street", "72 Street"]
corridor2 = ["72 Street", "80 Street", "98 Street", "Sub Avenue", "100 Street"]
corridor3 = ["100 Street", "53 Avenue", "19 Avenue", "11 Avenue"]

corridors = [corridor1, corridor2, corridor3]

# 3) Enumerate corridor-based routes (sub-paths).
#    Each route is a consecutive slice of a corridor, going forward only.
#    Critically, we only keep it if every adjacent pair is a valid link 
#    in the Travel times sheet (not '-' or NaN).

all_routes = []

for cor in corridors:
    n = len(cor)
    # Generate all consecutive sub-slices cor[start : end+1]
    for start in range(n):
        for end in range(start+1, n):
            candidate_stations = cor[start:end+1]  # e.g. [53 Street, SB Park, 66 Street]
            # Check adjacency for each consecutive pair
            valid = True
            for i in range(len(candidate_stations)-1):
                s1 = candidate_stations[i]
                s2 = candidate_stations[i+1]
                # If travel_time_df[s1,s2] is '-' or NaN, it's not a valid link
                if (pd.isnull(travel_time_df.loc[s1, s2]) or 
                    travel_time_df.loc[s1, s2] == '-'):
                    valid = False
                    break
            if valid:
                all_routes.append(candidate_stations)

# Remove duplicates (if any)
# convert to tuple then back to list to ensure uniqueness
all_routes = list({tuple(r): None for r in all_routes}.keys())
all_routes = [list(t) for t in all_routes]


# For each route, define its arcs ( station_i -> station_{i+1} )
#    Also define the route's cost = 1000 + 100*(# of stations).

route_arcs = {}
route_cost = {}

for r_idx, r_stations in enumerate(all_routes):
    arcs = []
    for i in range(len(r_stations)-1):
        u = r_stations[i]
        v = r_stations[i+1]
        arcs.append((u, v))
    route_arcs[r_idx] = arcs
    route_cost[r_idx] = 1000 + 100 * len(r_stations)


# model
#    Decision variables:
#      y[r] in {0,1}: route usage
#      b[r] in [0..5]: number of buses on route r
#      x[r, arc, (o,d)] >= 0: flow of OD=(o,d) on that arc of route r

model = gp.Model("Bus_Route_Design_with_Transfers")

y = {}
b = {}
x = {}

for r_idx in range(len(all_routes)):
    # Route usage
    y[r_idx] = model.addVar(vtype=GRB.BINARY, name=f"y_{r_idx}")
    b[r_idx] = model.addVar(vtype=GRB.INTEGER, lb=0, ub=5, name=f"b_{r_idx}")

    # Flow variables for each arc of route r, for each OD
    for arc in route_arcs[r_idx]:
        for (o, d) in demand:
            if demand[(o, d)] > 0:
                x[(r_idx, arc, (o, d))] = model.addVar(lb=0.0, vtype=GRB.CONTINUOUS,
                                                      name=f"x_{r_idx}_{arc}_{o}->{d}")

model.update()


# Flow-Balance Constraints:
#    For each station s, for each OD pair (o,d):
#      inflow + supply = outflow + demand
#
#    supply = D_{o,d} if s==o else 0
#    demand = D_{o,d} if s==d else 0
#
#    inflow(s) = sum of x[r,(u->s),(o,d)] over all routes r, arcs that end in s
#    outflow(s) = sum of x[r,(s->v),(o,d)] over all routes r, arcs that start in s

for s in stations:
    for (o, d) in demand:
        if o == d:
            continue
        supply = demand[(o, d)] if (s == o) else 0.0
        consume = demand[(o, d)] if (s == d) else 0.0

        inflow_expr = gp.LinExpr()
        outflow_expr = gp.LinExpr()

        # Sum flows that end at s
        for r_idx in range(len(all_routes)):
            for arc in route_arcs[r_idx]:
                if arc[1] == s and (r_idx, arc, (o, d)) in x:
                    inflow_expr.add(x[(r_idx, arc, (o, d))])

                if arc[0] == s and (r_idx, arc, (o, d)) in x:
                    outflow_expr.add(x[(r_idx, arc, (o, d))])

        model.addConstr(inflow_expr + supply == outflow_expr + consume,
                        name=f"flowbal_{s}_{o}->{d}")


# Capacity constraints on each arc:
# sum of x[r, arc, (o,d)] over all OD pairs <= 150 * b[r]

for r_idx in range(len(all_routes)):
    for arc in route_arcs[r_idx]:
        expr = gp.LinExpr()
        for (o, d) in demand:
            if (r_idx, arc, (o, d)) in x:
                expr.add(x[(r_idx, arc, (o, d))])
        model.addConstr(expr <= 150 * b[r_idx], name=f"arc_cap_{r_idx}_{arc}")


# Route usage logic:
#    b[r] <= 5 * y[r]
#    x[r,arc,(o,d)] <= demand[o,d] * y[r]
#    => If y[r] = 0, we cannot assign buses or flow to that route

for r_idx in range(len(all_routes)):
    model.addConstr(b[r_idx] <= 5 * y[r_idx], name=f"bus_logic_{r_idx}")

    for arc in route_arcs[r_idx]:
        for (o, d) in demand:
            if (r_idx, arc, (o, d)) in x:
                model.addConstr(x[(r_idx, arc, (o, d))] 
                                <= demand[(o, d)] * y[r_idx],
                                name=f"use_logic_{r_idx}_{arc}_{o}->{d}")


#Total bus limit: sum of b[r] <= 100

model.addConstr(gp.quicksum(b[r_idx] for r_idx in range(len(all_routes)))
                <= 100, 
                name="total_bus_limit")


# Objective

model.setObjective(
    gp.quicksum(route_cost[r_idx] * y[r_idx] for r_idx in range(len(all_routes))),
    GRB.MINIMIZE
)


# Solve

model.optimize()


if model.status == GRB.OPTIMAL:
    print(f"Optimal total cost = {model.ObjVal:.2f}")
    for r_idx in range(len(all_routes)):
        if y[r_idx].X > 0.5:
            print(f"Route {r_idx}: {all_routes[r_idx]}")
            print(f"   # of Buses: {b[r_idx].X}")
else:
    print(f"Infeasible or no optimal solution. Model status = {model.status}")



Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 5673 rows, 4444 columns and 22106 nonzeros
Model fingerprint: 0xfdd89f09
Variable types: 4400 continuous, 44 integer (22 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [1e+03, 2e+03]
  Bounds range     [1e+00, 5e+00]
  RHS range        [3e+01, 2e+02]
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.02 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -
Infeasible or no optimal solution. Model status = 3


In [91]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

# Read Excel data
excel_file = '/Users/juanjo/Library/CloudStorage/OneDrive-UniversityofOklahoma/1 2025 Spring/Optimization/Homeworks/HW1/Transportation data.xlsx'

# Clean travel times and OD matrix
def clean_df(df):
    return df.applymap(lambda x: pd.to_numeric(x, errors='coerce')).fillna(0)

travel_time_df = pd.read_excel(excel_file, sheet_name='Travel times', index_col=0)
travel_time_df = clean_df(travel_time_df)

od_df = pd.read_excel(excel_file, sheet_name='OD matrix', index_col=0)
od_df = clean_df(od_df)

# Extract stations and demand
stations = list(od_df.index)
demand = {(o, d): od_df.loc[o, d] for o in stations for d in stations if o != d and od_df.loc[o, d] > 0}

# Define corridors (as before)
corridor1 = ["53 Street", "SB Park", "66 Street", "72 Street"]
corridor2 = ["72 Street", "80 Street", "98 Street", "Sub Avenue", "100 Street"]
corridor3 = ["100 Street", "53 Avenue", "19 Avenue", "11 Avenue"]
corridors = [corridor1, corridor2, corridor3]

# Generate valid routes (as before)
all_routes = []
for cor in corridors:
    n = len(cor)
    for start in range(n):
        for end in range(start+1, n):
            candidate_stations = cor[start:end+1]
            valid = True
            for i in range(len(candidate_stations)-1):
                s1, s2 = candidate_stations[i], candidate_stations[i+1]
                if travel_time_df.loc[s1, s2] <= 0:
                    valid = False
                    break
            if valid:
                all_routes.append(candidate_stations)
all_routes = [list(t) for t in {tuple(r): None for r in all_routes}.keys()]

# Define route arcs and costs (as before)
route_arcs = {}
route_cost = {}
for r_idx, r_stations in enumerate(all_routes):
    arcs = [(r_stations[i], r_stations[i+1]) for i in range(len(r_stations)-1)]
    route_arcs[r_idx] = arcs
    route_cost[r_idx] = 1000 + 100 * len(r_stations)

# Model setup
model = gp.Model("Bus_Route_Design_with_Transfers")

y = model.addVars(len(all_routes), vtype=GRB.BINARY, name="y")
b = model.addVars(len(all_routes), vtype=GRB.INTEGER, lb=0, ub=5, name="b")

# Create flow variables only for OD pairs with demand
x = {}
od_pairs = [(o, d) for (o, d) in demand if demand[(o, d)] > 0]
for r_idx in range(len(all_routes)):
    for arc in route_arcs[r_idx]:
        for (o, d) in od_pairs:
            x_key = (r_idx, arc, (o, d))
            x[x_key] = model.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name=f"x_{x_key}")

model.update()

# Flow balance (as before)
for s in stations:
    for (o, d) in od_pairs:
        supply = demand[(o, d)] if s == o else 0.0
        consume = demand[(o, d)] if s == d else 0.0

        inflow = gp.quicksum(
            x.get((r_idx, arc, (o, d)), 0) 
            for r_idx in range(len(all_routes)) 
            for arc in route_arcs[r_idx] if arc[1] == s
        )
        outflow = gp.quicksum(
            x.get((r_idx, arc, (o, d)), 0) 
            for r_idx in range(len(all_routes)) 
            for arc in route_arcs[r_idx] if arc[0] == s
        )
        model.addConstr(inflow + supply == outflow + consume, f"flowbal_{s}_{o}->{d}")

# Capacity constraints (as before)
for r_idx in range(len(all_routes)):
    for arc in route_arcs[r_idx]:
        total_flow = gp.quicksum(x.get((r_idx, arc, od), 0) for od in od_pairs)
        model.addConstr(total_flow <= 150 * b[r_idx], f"arc_cap_{r_idx}_{arc}")

# Route usage logic (as before)
for r_idx in range(len(all_routes)):
    model.addConstr(b[r_idx] <= 5 * y[r_idx], f"bus_logic_{r_idx}")
    for arc in route_arcs[r_idx]:
        for (o, d) in od_pairs:
            if (r_idx, arc, (o, d)) in x:
                model.addConstr(x[(r_idx, arc, (o, d))] <= demand[(o, d)] * y[r_idx], f"use_logic_{r_idx}_{arc}_{o}->{d}")

model.addConstr(gp.quicksum(b[r_idx] for r_idx in range(len(all_routes))) <= 100, "total_bus_limit")

model.setObjective(gp.quicksum(route_cost[r_idx] * y[r_idx] for r_idx in range(len(all_routes))), GRB.MINIMIZE)

model.optimize()

# Output results (as before)
if model.status == GRB.OPTIMAL:
    print(f"Optimal total cost = {model.ObjVal:.2f}")
    for r_idx in range(len(all_routes)):
        if y[r_idx].X > 0.5:
            print(f"Route {r_idx}: {all_routes[r_idx]} (Buses: {b[r_idx].X})")
else:
    print("No optimal solution found.")

/var/folders/8b/2j_zssw92_j0mfff9tfhpygw0000gn/T/ipykernel_82175/3040155292.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(lambda x: pd.to_numeric(x, errors='coerce')).fillna(0)
/var/folders/8b/2j_zssw92_j0mfff9tfhpygw0000gn/T/ipykernel_82175/3040155292.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(lambda x: pd.to_numeric(x, errors='coerce')).fillna(0)


Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 5673 rows, 4444 columns and 22106 nonzeros
Model fingerprint: 0xf868182b
Variable types: 4400 continuous, 44 integer (22 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [1e+03, 2e+03]
  Bounds range     [1e+00, 5e+00]
  RHS range        [3e+01, 2e+02]
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -
No optimal solution found.


# Question 3

In [ ]:
# a

In [ ]:
# b

In [1]:
from gurobipy import Model, GRB
import pandas as pd

def solve_bishop_problem(N):
    model = Model("MaxIndependentBishops")

    # Decision variables: x[r,c] = 1 if a bishop is placed at (r, c)
    x = model.addVars(N, N, vtype=GRB.BINARY, name="x")

    # Constraint 1: No two bishops on the same main diagonal (r+c)
    for d in range(2 * N - 1):
        model.addConstr(sum(x[r, c] for r in range(N) for c in range(N) if r + c == d) <= 1)

    # Constraint 2: No two bishops on the same anti-diagonal (r-c)
    for d in range(-N + 1, N):
        model.addConstr(sum(x[r, c] for r in range(N) for c in range(N) if r - c == d) <= 1)

    # Objective function: Maximize the number of bishops
    model.setObjective(sum(x[r, c] for r in range(N) for c in range(N)), GRB.MAXIMIZE)

    # Solve the model
    model.optimize()

    # Extract results
    board = [[0] * N for _ in range(N)]
    for r in range(N):
        for c in range(N):
            if x[r, c].X > 0.5:
                board[r][c] = 1  # Bishop placed

    return board, model.objVal

# Solve for N = 6, 8, 10
solutions = {N: solve_bishop_problem(N) for N in [6, 8, 10]}

# Display solutions
for N, (board, max_bishops) in solutions.items():
    df = pd.DataFrame(board, index=[f"R{i+1}" for i in range(N)], columns=[f"C{j+1}" for j in range(N)])
    print(f"\nMaximum number of bishops for {N}x{N} board: {max_bishops}")
    print(df)

Set parameter Username
Academic license - for non-commercial use only - expires 2025-08-19
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 22 rows, 36 columns and 72 nonzeros
Model fingerprint: 0x485acf76
Variable types: 0 continuous, 36 integer (36 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 8.0000000
Presolve removed 14 rows and 24 columns
Presolve time: 0.00s
Presolved: 8 rows, 12 columns, 24 nonzeros
Found heuristic solution: objective 9.0000000
Variable types: 0 continuous, 12 integer (12 binary)

Root relaxation: objective 1.000000e+01, 4 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  